In [1]:
!nvidia-smi

Thu Oct  8 07:12:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
from numba import cuda

device = cuda.get_current_device()

print("Device Name:", device.name)
print("Compute Capability:", device.compute_capability)
print("Max Threads Per Block:", device.MAX_THREADS_PER_BLOCK)
print("Warp Size:", device.WARP_SIZE)
print("Max Block Dimensions:",
      device.MAX_BLOCK_DIM_X,
      device.MAX_BLOCK_DIM_Y,
      device.MAX_BLOCK_DIM_Z)


Device Name: Tesla T4
Compute Capability: ComputeCapability(major=7, minor=5)
Max Threads Per Block: 1024
Warp Size: 32
Max Block Dimensions: 1024 1024 64


In [4]:
import numpy as np
import time
from numba import cuda

def double_cpu_pure(arr):
    out = np.empty_like(arr)
    for i in range(len(arr)):
        out[i] = arr[i] * 2
    return out

@cuda.jit
def double_gpu_kernel(inp, out):
    idx = cuda.grid(1)
    if idx < inp.size:
        out[idx] = inp[idx] * 2

In [5]:
def benchmark(N):

    h = np.random.rand(N).astype(np.float32)

    if N <= 1000000:
        t = time.perf_counter()
        double_cpu_pure(h)
        cpu = (time.perf_counter()-t)*1000
    else:
        cpu = None

    threads = 256
    blocks = (N+threads-1)//threads

    d = cuda.to_device(h)
    dout = cuda.device_array_like(d)

    double_gpu_kernel[blocks,threads](d,dout)
    cuda.synchronize()

    t = time.perf_counter()

    d = cuda.to_device(h)
    dout = cuda.device_array_like(d)

    double_gpu_kernel[blocks,threads](d,dout)
    cuda.synchronize()

    dout.copy_to_host()

    gpu_total = (time.perf_counter()-t)*1000

    d = cuda.to_device(h)
    dout = cuda.device_array_like(d)

    t = time.perf_counter()

    double_gpu_kernel[blocks,threads](d,dout)
    cuda.synchronize()

    gpu_kernel = (time.perf_counter()-t)*1000

    return cpu,gpu_total,gpu_kernel

In [6]:
sizes = [
    100,
    1000,
    10000,
    100000,
    1000000,
    5000000,
    10000000
]

for n in sizes:

    cpu,gpu,gkernel = benchmark(n)

    print(n,cpu,gpu,gkernel)

/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


100 0.0909789999923305 3.284057999962897 0.2246110000214685
1000 0.4820450000124765 0.7857729999614094 0.21988499997860345
10000 20.28763299995262 0.7117210000160412 0.19090299997515103


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 40 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


100000 195.23643400009405 8.103257000016129 0.22959999989780044
1000000 606.7886100000806 2.4948390000645304 0.18667999995614082
5000000 None 11.800182999991193 0.3344049999896015
10000000 None 23.886198000013792 0.520137999956205


In [7]:
@cuda.jit
def defective_kernel(arr):

    idx = cuda.grid(1)

    arr[idx]=999.0

In [8]:
N=1000

arr=np.zeros(N,dtype=np.float32)

d=cuda.to_device(arr)

threads=256
blocks=4

try:

    defective_kernel[blocks,threads](d)

    cuda.synchronize()

    print(d.copy_to_host()[-5:])

except Exception as e:

    print(e)

[999. 999. 999. 999. 999.]


In [9]:
import numpy as np
from numba import cuda


@cuda.jit
def naive_sum_kernel(arr, total):
    idx = cuda.grid(1)

    if idx < arr.size:
        total[0] += arr[idx]


N = 50000

data = np.ones(N, dtype=np.float32)

d_data = cuda.to_device(data)


threads = 256
blocks = (N + threads - 1) // threads


for trial in range(5):

    d_total = cuda.to_device(np.array([0.0], dtype=np.float32))

    naive_sum_kernel[blocks, threads](d_data, d_total)

    cuda.synchronize()

    result = d_total.copy_to_host()[0]

    deviation = 50000.0 - result

    print(
        f"Trial {trial+1}: "
        f"Output = {result}, "
        f"Lost Updates = {deviation}"
    )

Trial 1: Output = 2.0, Lost Updates = 49998.0
Trial 2: Output = 2.0, Lost Updates = 49998.0
Trial 3: Output = 2.0, Lost Updates = 49998.0
Trial 4: Output = 2.0, Lost Updates = 49998.0
Trial 5: Output = 2.0, Lost Updates = 49998.0


In [18]:
import hashlib

def generate_lab_checksum(student_id_str, d_canvas, exp_table_crossover):
    h = hashlib.sha256()
    h.update(student_id_str.strip().encode('utf-8'))

    # Compatibility fix for current Numba version
    h.update(cuda.get_current_device().name.encode('utf-8'))

    h.update(d_canvas.copy_to_host()[:10, :10].tobytes())
    h.update(str(exp_table_crossover).encode('utf-8'))

    digest = h.hexdigest()[:16].upper()

    print("=" * 60)
    print(f"OFFICIAL VERIFICATION CHECKSUM TOKEN: {digest}")
    print("=" * 60)

    return digest

STUDENT_ID = "230103346"
CROSSOVER_N = 10000

generate_lab_checksum(STUDENT_ID, d_canvas, CROSSOVER_N)

NameError: name 'd_canvas' is not defined